In [1]:
# Download Dataset
import os
import re
import tarfile
import pandas as pd

print('Downloading dataset (~30MB)...')
!wget -q https://www.cs.jhu.edu/~mdredze/datasets/sentiment/domain_sentiment_data.tar.gz \
     -O /content/domain_sentiment_data.tar.gz

with tarfile.open('/content/domain_sentiment_data.tar.gz', 'r:gz') as tar:
    tar.extractall('/content/')

print(f'Done. Folders found: {os.listdir("/content/sorted_data_acl/")}')


/tmp/ipykernel_5802/784309849.py:11: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('/content/')


Done. Folders found: ['books', 'kitchen_&_housewares', 'electronics', 'dvd']


In [2]:
# Install HuggingFace
!pip install transformers -q

In [3]:
# Imports
import os
import re
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Training on: {device}")

Training on: cuda


In [4]:
# Load Reviews
data_path  = '/content/sorted_data_acl/'
categories = ['books', 'dvd', 'electronics', 'kitchen_&_housewares']

def parse_review_file(filepath, label):
    reviews = []
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        content = f.read()
    matches = re.findall(r'<review_text>(.*?)</review_text>', content, re.DOTALL)
    for text in matches:
        text = text.strip()
        if text:
            reviews.append({'text': text, 'label': label})
    return reviews

all_reviews = []
for category in categories:
    pos_path = os.path.join(data_path, category, 'positive.review')
    neg_path = os.path.join(data_path, category, 'negative.review')
    if os.path.exists(pos_path):
        parsed = parse_review_file(pos_path, 1)
        all_reviews.extend(parsed)
        print(f'{category} - positive: {len(parsed)} reviews')
    if os.path.exists(neg_path):
        parsed = parse_review_file(neg_path, 0)
        all_reviews.extend(parsed)
        print(f'{category} - negative: {len(parsed)} reviews')

df = pd.DataFrame(all_reviews)
print(f'\nTotal reviews: {len(df)}')
print(f'Positive: {df["label"].sum()} | Negative: {(df["label"] == 0).sum()}')


books - positive: 1000 reviews
books - negative: 1000 reviews
dvd - positive: 1000 reviews
dvd - negative: 1000 reviews
electronics - positive: 1000 reviews
electronics - negative: 1000 reviews
kitchen_&_housewares - positive: 1000 reviews
kitchen_&_housewares - negative: 1000 reviews

Total reviews: 8000
Positive: 4000 | Negative: 4000


In [5]:
# Outlier removal (reviews less than 5 words and more than 500 words) + Shuffle
df['word_count'] = df['text'].apply(lambda x: len(x.split()))
df = df[(df['word_count'] >= 5) & (df['word_count'] <= 500)]
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"After cleaning: {len(df)} reviews")
print(f"Positive: {df['label'].sum()} | Negative: {(df['label'] == 0).sum()}")

After cleaning: 7753 reviews
Positive: 3856 | Negative: 3897


In [6]:
# Load BERT Tokeniser - BERT uses raw text, not cleaned text
# It handles its own tokenisation internally
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print("BERT tokeniser loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BERT tokeniser loaded.


In [7]:
# Create Dataset Class
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length     = self.max_len,
            padding        = 'max_length',
            truncation     = True,
            return_tensors = 'pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [8]:
# Train/Validation/Test Split + DataLoaders
texts  = df['text'].tolist()
labels = df['label'].tolist()

train_end = int(len(texts) * 0.8)
val_end   = int(len(texts) * 0.9)

train_dataset = SentimentDataset(texts[:train_end],       labels[:train_end],       tokenizer)
val_dataset   = SentimentDataset(texts[train_end:val_end], labels[train_end:val_end], tokenizer)
test_dataset  = SentimentDataset(texts[val_end:],         labels[val_end:],         tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=16, shuffle=False)

print(f"Train: {len(train_dataset)} | Validation: {len(val_dataset)} | Test: {len(test_dataset)}")
print(f"Train batches: {len(train_loader)} | Validation batches: {len(val_loader)}")

Train: 6202 | Validation: 775 | Test: 776
Train batches: 388 | Validation batches: 49


In [9]:
# Load BERT Model
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels = 2
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"BERT model loaded.")
print(f"Total parameters: {total_params:,}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT model loaded.
Total parameters: 109,483,778


In [10]:
#  Train BERT
epochs    = 3
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

total_steps = len(train_loader) * epochs
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = total_steps // 10,
    num_training_steps = total_steps
)

best_val_acc     = 0
best_model_state = None

for epoch in range(epochs):
    # Training
    model.train()
    train_correct = 0
    train_total   = 0
    train_losses  = []

    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        model.zero_grad()
        outputs = model(input_ids      = input_ids,
                        attention_mask = attention_mask,
                        labels         = labels)

        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        preds          = torch.argmax(outputs.logits, dim=1)
        train_correct += (preds == labels).sum().item()
        train_total   += labels.size(0)
        train_losses.append(loss.item())

    # Validation
    model.eval()
    val_correct = 0
    val_total   = 0

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs        = model(input_ids      = input_ids,
                                   attention_mask = attention_mask)
            preds          = torch.argmax(outputs.logits, dim=1)
            val_correct   += (preds == labels).sum().item()
            val_total     += labels.size(0)

    train_acc = train_correct / train_total * 100
    val_acc   = val_correct   / val_total   * 100
    avg_loss  = sum(train_losses) / len(train_losses)

    print(f"Epoch {epoch+1}/{epochs} | "
          f"Train Loss: {avg_loss:.4f} | "
          f"Train Acc: {train_acc:.2f}% | "
          f"Validation Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc     = val_acc
        best_model_state = model.state_dict().copy()
        print(f"  Best model saved (val acc: {best_val_acc:.2f}%)")

model.load_state_dict(best_model_state)
print("\nBest model restored for testing.")

Epoch 1/3 | Train Loss: 0.3872 | Train Acc: 81.55% | Validation Acc: 88.00%
  Best model saved (val acc: 88.00%)
Epoch 2/3 | Train Loss: 0.1836 | Train Acc: 94.02% | Validation Acc: 91.48%
  Best model saved (val acc: 91.48%)
Epoch 3/3 | Train Loss: 0.0939 | Train Acc: 97.58% | Validation Acc: 91.10%

Best model restored for testing.


In [14]:
# Test Evaluation
model.eval()
test_correct = 0
test_total   = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        outputs        = model(input_ids      = input_ids,
                               attention_mask = attention_mask)
        preds          = torch.argmax(outputs.logits, dim=1)
        test_correct  += (preds == labels).sum().item()
        test_total    += labels.size(0)

test_accuracy = test_correct / test_total * 100
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Accuracy: 90.72%


In [12]:
# Inference Function
def predict_sentiment_bert(text, model, tokenizer):
    model.eval()
    encoding = tokenizer(
        text,
        max_length     = 256,
        padding        = 'max_length',
        truncation     = True,
        return_tensors = 'pt'
    )
    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=1)
        pred    = torch.argmax(probs, dim=1).item()
        conf    = probs[0][pred].item()

    label = "Positive review" if pred == 1 else "Negative review"
    print(f"Prediction : {label}")
    print(f"Confidence : {conf:.4f}")
    return label

# Test both directions
predict_sentiment_bert("This product is absolutely amazing, I love it!", model, tokenizer)
predict_sentiment_bert("Terrible quality, broke after one day. Complete waste of money.", model, tokenizer)

Prediction : Positive review
Confidence : 0.9982
Prediction : Negative review
Confidence : 0.9982


'Negative review'

In [15]:
# Save BERT Model
bert_save_path = '/content/models/bert_finetuned/'
os.makedirs(bert_save_path, exist_ok=True)

model.save_pretrained(bert_save_path)
tokenizer.save_pretrained(bert_save_path)

print(f'BERT model saved to {bert_save_path}')
print('Files saved:', os.listdir(bert_save_path))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BERT model saved to /content/models/bert_finetuned/
Files saved: ['model.safetensors', 'config.json', 'tokenizer_config.json', 'tokenizer.json']
